# 第22章 分组、聚合与数据透视

使用groupby、agg、transform、pivot_table和crosstab回答分组问题。


## 先解决一个小问题

拿一组小型业务数据练习“分组、聚合与数据透视”：先看数据结构，再完成一次明确的计算或转换。使用groupby、agg、transform、pivot_table和crosstab回答分组问题。


## 这章为什么先学

这是“Pandas”路线中第 22 章的操作重点。本章只解决“分组、聚合与数据透视”，不重复前面章节已经完成的准备工作。


## 开始前确认

- 掌握 Python 基础语法、列表和字典
- 开始前先确认：执行分组聚合


## 做完要留下什么

产出一个与“分组、聚合与数据透视”直接对应的结果，并记录输入形状、字段或筛选口径。


## 运行规则

代码单元格按依赖顺序执行；需要复现结果时从上到下运行，并保留输入、计算和输出。


## 本章要会

- 执行分组聚合
- 一次计算多个指标
- 保留原行的组内计算
- 构建透视表和交叉表


## 核心概念

- 分组前必须明确维度、指标和聚合函数。
- agg压缩行数，transform保持原行数。
- 透视表中的缺失组合与真实0含义不同。


## 示例 1：分组聚合

命名聚合让输出列直接表达口径。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南", "华北", "华北"],
    "channel": ["线上", "线下", "线上", "线下", "线上", "线下"],
    "amount": [520, 310, 460, 280, 390, 260],
    "quantity": [3, 2, 2, 1, 2, 1],
})
summary = orders.groupby("region").agg(
    sales=("amount", "sum"),
    orders=("amount", "size"),
    average=("amount", "mean"),
)
print(summary)


## 示例 2：transform组内占比

transform结果与原表等长，可直接添加为新列。


In [ ]:
orders["region_total"] = orders.groupby("region")["amount"].transform("sum")
orders["region_share"] = orders["amount"] / orders["region_total"]
print(orders)


## 示例 3：透视表与交叉表

index与columns分别定义行维度和列维度。


In [ ]:
pivot = orders.pivot_table(
    index="region", columns="channel", values="amount", aggfunc="sum", fill_value=0
)
counts = pd.crosstab(orders["region"], orders["channel"], margins=True)
print(pivot)
print(counts)


## 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd
from js import window

# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = f"{window.location.origin}/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates=["InvoiceDate"],
    dtype={"InvoiceNo": "string", "StockCode": "string", "Description": "string", "Country": "category"},
).rename(columns={
    "InvoiceNo": "order_id", "StockCode": "stock_code", "Description": "description",
    "Quantity": "quantity", "InvoiceDate": "order_time", "UnitPrice": "unit_price",
    "CustomerID": "customer_id", "Country": "country",
})
large_orders["sales"] = (large_orders["quantity"] * large_orders["unit_price"]).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C") | (large_orders["quantity"] < 0),
    "取消/退货", "完成"
)
print(f"UCI Online Retail 公开数据：{len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print("内存占用：", f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
large_orders.head()


In [ ]:
summary = (
    large_orders.query("status == '完成'")
    .groupby([large_orders["order_time"].dt.to_period("M"), "country"], observed=True)
    .agg(销售额=("sales", "sum"), 订单数=("order_id", "size"), 客单价=("sales", "mean"))
    .reset_index()
)
pivot = summary.pivot(index="order_time", columns="country", values="销售额")
print(f"聚合前 {len(large_orders):,} 行，聚合后 {len(summary):,} 行")
display(summary.head(10))
display(pivot.tail().round(0))


## 常见误区

- 使用mean却把结果描述为合计
- groupby后忘记处理索引
- 把缺失组合无条件填0


## 综合练习

1. 按地区和渠道分组
2. 计算销售额、订单数和平均订单金额
3. 构建地区×渠道透视表

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“按地区和渠道分组”。
2. **独立完成**：不复制示例代码，完成“计算销售额、订单数和平均订单金额”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“构建地区×渠道透视表”，用一两句话说明你修改了什么。

### 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南", "华北"],
    "channel": ["广告", "自然", "广告", "自然", "广告"],
    "amount": [620, 410, 530, 380, 470],
})

# TODO: 按地区和渠道分组，计算销售额、订单数和平均订单金额
summary = orders.groupby(["region", "channel"]).agg(
    sales=("amount", "sum"),
    order_count=("amount", "size"),
    average_order=("amount", "mean"),
).reset_index()

# TODO: 构建地区×渠道透视表（销售额）
pivot =

print(summary)
print(pivot)


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南", "华北"],
    "channel": ["广告", "自然", "广告", "自然", "广告"],
    "amount": [620, 410, 530, 380, 470],
})
summary = orders.groupby(["region", "channel"]).agg(
    sales=("amount", "sum"),
    order_count=("amount", "size"),
    average_order=("amount", "mean"),
).reset_index()
pivot = summary.pivot(index="region", columns="channel", values="sales").fillna(0)
print(summary)
print(pivot)

# 自检
assert len(summary) == 4, "检查分组结果：应该有4个组合"
assert pivot.loc["华东", "广告"] == 620, "检查透视表：华东-广告应该是620"


## 本章小结

使用groupby、agg、transform、pivot_table和crosstab回答分组问题。

**迁移思考**：

1. 如果需要计算每个地区销售额占全国的比例，应该用 agg 还是 transform？为什么？
2. 为什么透视表中的缺失组合不能无条件填0？什么情况下填0是合理的？


### 你已经掌握

- 执行分组聚合
- 一次计算多个指标
- 保留原行的组内计算
- 构建透视表和交叉表


### 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 分组聚合 | 命名聚合让输出列直接表达口径。 | `pd.DataFrame()`、`orders.groupby()`、`.agg()` |
| transform组内占比 | transform结果与原表等长，可直接添加为新列。 | `orders.groupby()`、`.transform()`、`orders["region_total"]`、`orders["region_share"]` |
| 透视表与交叉表 | index与columns分别定义行维度和列维度。 | `orders.pivot_table()`、`pd.crosstab()`、`orders["region"]`、`orders["channel"]` |


### 需要注意

- 使用mean却把结果描述为合计
- groupby后忘记处理索引
- 把缺失组合无条件填0


### 完成检查

- [ ] 能够执行分组聚合
- [ ] 能够一次计算多个指标
- [ ] 能够保留原行的组内计算
- [ ] 能够构建透视表和交叉表


### 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
